In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('delta').load('abfss://olympics@adls7428.dfs.core.windows.net/bronze/athletes')
df.display()

In [0]:
df = df.fillna({"birth_place":"xyz","birth_country":"xyz","residence_place":"xyz","residence_country":"aaa","residence_place":"unknown"})
df.display()

In [0]:
df_filtered = df.where((col('current')== True) & col('name').isin('GALSTYAN Slavik','HARUTYUNYAN Arsen','CAMARA Ebrahima'))
df_filtered.display()

In [0]:
df_sorted = df.sort('height','weight',ascending=[1,0]).where(col('weight')>0)
df_sorted.display()

In [0]:
df_data = df_sorted.withColumn('nationality',regexp_replace(col('nationality'),'United States','US'))
df_data.display()

In [0]:
df_unique = df_sorted.groupBy('code').agg(count('code').alias('total_count')).where(col('total_count')>1)
df_unique.display()

In [0]:
df_final = df_sorted.withColumnRenamed('code','athlete_id')
df_final.display()

In [0]:
df_final = df_final.withColumn('occupation',split(col('occupation'),','))
df_final.display()

In [0]:
df_final.columns

In [0]:
df_window = df_final.select('athlete_id',
 'current',
 'name',
 'name_short',
 'name_tv',
 'gender',
 'function',
 'country_code',
 'country',
 'country_long',
 'nationality',
 'nationality_long',
 'nationality_code',
 'height',
 'weight')



In [0]:
display(df_window)

Databricks visualization. Run in Databricks to view.

In [0]:
df_window.withColumn('cum_weight',sum('weight').over(Window.partitionBy('nationality').orderBy(desc('height')).rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

In [0]:
df_final.write.format('delta').mode('append').option('path','abfss://olympics@adls7428.dfs.core.windows.net/silver/athletes').saveAsTable('olympics.silver.silver_athletes')